[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/18_escala_muestreo_coste/18_proyecto_escala_muestreo_coste.ipynb)

# Proyecto 18 · Diseña la política de muestreo, cardinalidad y retención de SupportOps (con presupuesto) y mide su impacto

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 2 h 30 min |
| **Servicios de Compose** | Ninguno. Con Docker, los pasos 6 y 9 lanzan contenedores efímeros (Collector, Loki); sin Docker se evalúan solo por texto |
| **RAM aproximada** | < 1 GB |
| **Prerrequisitos** | Lección `18_escala_muestreo_coste`; proyectos 05 y 16 |
| **Conexión** | Aplica el tail sampling que el módulo 05 dejó fuera y protege los SLO del 16 (nada interesante se pierde); el resultado es el Collector de dos capas que se despliega en el módulo 19 |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El agente pasa de piloto a producción (2 M tickets/día) y finanzas pone un tope: **las trazas conservadas no pueden pasar del 25 % de los bytes generados**, sin perder ninguna traza con error, lenta o cara,
sin falsear la tasa de error de los paneles, con el Collector dimensionado para que no descarte trazas y con un presupuesto mensual de almacenamiento. Además, debes impedir que un servicio ruidoso (etiquetas con ids, bucles de logs, trazas gigantes) tumbe a los demás.

## Datos

Corpus de 1 500 tickets (mock, semilla 42) generado en el notebook; cada traza lleva su tamaño OTLP real. Sin descargas. Con Docker puedes validar tu configuración con el Collector y Loki reales.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `volume_report` | spans/traza, bytes/span, fracción interesante y GB/día | ±1 % de lo medido; fracción de bytes interesante ∈ (10 %; 18 %) |
| R2 | `baseline_for_budget` | porcentaje base que cumple el 25 % de bytes | 11-15 % (la solución: 13,x %); con presupuesto del 5 % → 0 |
| R3 | `build_policies` | recall de interesantes y bytes conservados | 100 % y ≤ 27 % |
| R4 | `num_traces_for`, `evaluate` | dimensionado sin descartes (16× tráfico, 30 s) | `dropped_too_early` = 0; pico ≤ `num_traces` ≤ 2,2 × pico; `num_traces_for(200, 30)` = 9 000 |
| R5 | `corrected_error_rate` | tasa de error con pesos 1/π | error relativo < 15 %; la cruda > 3× la real (≈ 4,5×) |
| R6 | `collector_yaml` | `tail_sampling` + `span_metrics` + `forward`, OTTL para el coste, sin `numeric_attribute` | texto correcto y `otelcol validate` OK (con Docker) |
| R7 | `cardinality_check` | acepta lo sano y rechaza la tormenta señalando la etiqueta | exacto |
| R8 | `retention_plan` | días máximos dentro de un presupuesto (4 casos) | exacto |
| R9 | `tenant_limits` (con Docker) | Loki acepta 10 streams y rechaza 40 (429); Tempo con tope de bytes por traza ≤ 5 000 | exacto |

> 📏 Los umbrales salen de la solución de referencia con la semilla 42 fija; las comparaciones usan ratios y rangos, no tiempos absolutos.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import copy, collections, gzip, json, math, re, subprocess, time
from IPython.display import Markdown, display
from supportops import sampling as S
DOCKER = S.docker_ok()
REC = json.load(gzip.open(ROOT / "18_escala_muestreo_coste" / "recorded" / "capture_seed42.json.gz", "rt", encoding="utf-8"))   # modo degradado: medido con Docker real
CORPUS = S.build_corpus(1500, seed=SEED)                  # 1 500 tickets de SupportOps (mock, semilla 42) con su tamaño OTLP real
SM = CORPUS.summary()
print(f"Docker disponible: {DOCKER} (si es False, las secciones con contenedores usan la captura pregrabada) · {int(SM['traces'])} trazas, {int(SM['spans'])} spans, {SM['bytes_per_span']:.0f} B/span")

## Paso 0 · El presupuesto (te lo damos hecho)

El corpus, el tráfico objetivo (2 M tickets/día) y el presupuesto. `REF_RET` son las respuestas de referencia del paso 8 (para la rúbrica).

In [ ]:
TICKETS_DAY = 2_000_000
BUDGET_BYTES_PCT = 25.0
BUDGET_USD_MONTH = 40.0
REF_RET = {}
for g, b in ((100, 300), (100, 1200), (20, 200), (500, 100)):
    REF_RET[(g, b)] = max([d for d in range(1, 366) if g * (min(d, 7) * 0.10 + max(0, d - 7) * 0.023) <= b] or [0])
print(f"{len(CORPUS)} trazas · interesantes: {SM['interesting_rate']:.1%} · bytes por traza: {SM['bytes_per_trace']:.0f} · presupuesto: ≤ {BUDGET_BYTES_PCT} % de los bytes y {BUDGET_USD_MONTH} USD/mes")

## Paso 1 · El volumen (TODO)

Escribe `volume_report(corpus, tickets_per_day)` → dict con `spans_per_trace`, `bytes_per_span`, `interesting_rate`, `interesting_bytes_frac` (fracción de los **bytes** que pertenecen a trazas interesantes) y `gb_per_day_traces` (spans × bytes sin muestrear). Usa `corpus.summary()` y `Trace.interesting`.

<details>
<summary>🪜 Pista 1</summary>

`corpus.summary()` ya trae spans por traza y bytes por span; la fracción de bytes interesantes es `sum(t.bytes for t in corpus.traces if t.interesting) / corpus.total_bytes`.

</details>

<details>
<summary>🪜 Pista 2</summary>

GB/día = tickets_por_día × spans_por_traza × bytes_por_span / 1e9.

</details>

In [ ]:
def volume_report(corpus, tickets_per_day):
    # TODO
    raise NotImplementedError("TODO 1")

try:
    print(volume_report(CORPUS, TICKETS_DAY))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 2 · El mando del presupuesto (TODO)

Escribe `baseline_for_budget(corpus, budget_bytes_pct)`: el porcentaje de muestreo base (`probabilistic`) tal que **lo interesante (siempre conservado) + base × resto** no supere el presupuesto. Redondea hacia abajo a 1 decimal; si el presupuesto no alcanza ni para lo interesante, devuelve 0.

<details>
<summary>🪜 Pista 1</summary>

Es una ecuación lineal: despeja b de $i + b(1-i) \le B$.

</details>

<details>
<summary>🪜 Pista 2</summary>

`b = (B/100 - i) / (1 - i)`; el resultado en % con `math.floor(b*1000)/10`; `max(0, ...)`.

</details>

In [ ]:
def baseline_for_budget(corpus, budget_bytes_pct):
    # TODO: i + b·(1 − i) ≤ presupuesto, con i = fracción de bytes interesantes
    raise NotImplementedError("TODO 2")

try:
    print(baseline_for_budget(CORPUS, BUDGET_BYTES_PCT), baseline_for_budget(CORPUS, 5))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 3 · Las políticas (TODO)

Escribe `build_policies(baseline_pct)` → lista de políticas con **esos nombres**: `errors` (`StatusCode`), `slow` (`Latency` ≥ 5 000 ms), `costly` (`AttrCompare` sobre `supportops.cost_usd` > 0,0005; **no** `NumericAttribute`: recuerda la trampa del int64) y `baseline` (`Probabilistic`).

<details>
<summary>🪜 Pista 1</summary>

Cada clase de `S` tiene los mismos parámetros que la clave del Collector (`threshold_ms`, `status_codes`, `sampling_percentage`).

</details>

<details>
<summary>🪜 Pista 2</summary>

`S.StatusCode(("ERROR",), name="errors")`, `S.Latency(5000, name="slow")`, `S.AttrCompare("supportops.cost_usd", ">", 0.0005, name="costly")`, `S.Probabilistic(pct, name="baseline")`.

</details>

In [ ]:
def build_policies(baseline_pct):
    # TODO: S.StatusCode, S.Latency, S.AttrCompare, S.Probabilistic (con name=...)
    raise NotImplementedError("TODO 3")

try:
    pols = build_policies(baseline_for_budget(CORPUS, BUDGET_BYTES_PCT))
    c = S.coverage(S.TailSampler(pols, 30).run(CORPUS.traces).kept, CORPUS)
    print({k: round(v, 1) for k, v in c.items()})
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 4 · El dimensionado (TODO)

Escribe `num_traces_for(traces_per_s, decision_wait_s, margin=1.5)` y `evaluate(policies, corpus, decision_wait_s, rate_factor)`, que simula el tráfico ×`rate_factor` (`S.rescale_rate`), mide el pico de trazas en memoria con memoria ilimitada y vuelve a ejecutar con tu `num_traces`. Devuelve `S.coverage(...)` más `peak_traces`, `num_traces`, `dropped_too_early` y `kept`.

<details>
<summary>🪜 Pista 1</summary>

`traces_per_s` del corpus reescalado = `rate_factor × len(corpus) / corpus.duration_s`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`S.TailSampler(policies, decision_wait_s, num_traces=10**9).run(rescaled).peak_traces` da el pico sin límite; `num_traces_for(...)` ≥ pico.

</details>

In [ ]:
def num_traces_for(traces_per_s, decision_wait_s, margin=1.5):
    # TODO: trazas en memoria ≈ tráfico × espera; añade margen y redondea hacia arriba
    raise NotImplementedError("TODO 4a")

def evaluate(policies, corpus, decision_wait_s=30.0, rate_factor=16):
    # TODO: rescaled = S.rescale_rate(corpus.traces, rate_factor); S.TailSampler(...).run(...) -> .peak_traces, .kept, .dropped_too_early
    raise NotImplementedError("TODO 4b")

try:
    ev = evaluate(build_policies(12.0), CORPUS)
    print({k: v for k, v in ev.items() if k != "kept"})
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 5 · No falsees las métricas (TODO)

Escribe `corrected_error_rate(kept, baseline_pct)`: la tasa de error estimada sobre las trazas conservadas **con pesos de inclusión** (lo interesante pesa 1; el resto, 1/baseline). Usa `S.inclusion_weights` y `S.estimate_metrics(kept, pesos)`.

<details>
<summary>🪜 Pista 1</summary>

Una traza «interesante» (error, lenta o cara) se conserva siempre (π = 1); las demás con probabilidad `baseline_pct/100`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`keep_all = lambda t: t.error or t.slow or t.costly`; `w = S.inclusion_weights(kept, keep_all, baseline_pct/100)`.

</details>

In [ ]:
def corrected_error_rate(kept, baseline_pct):
    # TODO
    raise NotImplementedError("TODO 5")

try:
    kept = S.TailSampler(build_policies(12.0), 30).run(CORPUS.traces).kept
    print("verdad:", round(S.estimate_metrics(CORPUS.traces)["error_rate"], 4), "· cruda:", round(S.estimate_metrics(kept)["error_rate"], 4), "· corregida:", round(corrected_error_rate(kept, 12.0), 4))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 6 · La configuración del Collector (TODO)

Escribe `collector_yaml(policies, num_traces)`: la configuración completa (con `span_metrics` **antes** del muestreo, `forward`, `tail_sampling` con `decision_wait: 30s`). Con Docker, la rúbrica la valida con `otelcol-contrib validate`.

<details>
<summary>🪜 Pista 1</summary>

`supportops.sampling.collector_config` genera exactamente la estructura de la lección §6.

</details>

<details>
<summary>🪜 Pista 2</summary>

`S.collector_config(policies, decision_wait="30s", num_traces=num_traces)`.

</details>

In [ ]:
def collector_yaml(policies, num_traces):
    # TODO: S.collector_config(policies, decision_wait="30s", num_traces=num_traces)
    raise NotImplementedError("TODO 6")

try:
    Y = collector_yaml(build_policies(12.0), 9000)
    print(Y[:700])
    if DOCKER:
        print("otelcol validate →", S.validate_collector_config(Y)[0])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 7 · Cardinalidad (TODO)

Escribe `cardinality_check(dims, max_series)` → `(True, None)` si el producto de cardinalidades no supera `max_series`; si lo supera, `(False, etiqueta_con_mayor_cardinalidad)` (la que hay que quitar).

<details>
<summary>🪜 Pista 1</summary>

El nº de series es el producto de las cardinalidades.

</details>

<details>
<summary>🪜 Pista 2</summary>

`max(dims, key=dims.get)` devuelve la clave con el valor más alto.

</details>

In [ ]:
def cardinality_check(dims, max_series):
    # TODO: S.series_estimate(dims) y max(dims, key=dims.get)
    raise NotImplementedError("TODO 7")

try:
    print(cardinality_check({"intent": 4, "tier": 3, "model": 10}, 1000), cardinality_check({"intent": 4, "tier": 3, "ticket_id": 2_000_000}, 100_000))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 8 · Retención dentro del presupuesto (TODO)

Escribe `retention_plan(traces_gb_day, budget_usd_month, hot_price=0.10, cold_price=0.023, hot_days=7)` → el **máximo de días** de retención (1-365) cuyo coste mensual en régimen estacionario no supera el presupuesto: `gb × (min(d, hot_days)·hot_price + max(0, d − hot_days)·cold_price)`.

<details>
<summary>🪜 Pista 1</summary>

Prueba cada d de 1 a 365 y quédate con el mayor que cumple (el coste crece con d).

</details>

<details>
<summary>🪜 Pista 2</summary>

`best = max([d for d in range(1, 366) if coste(d) <= budget] or [0])`.

</details>

In [ ]:
def retention_plan(traces_gb_day, budget_usd_month, hot_price=0.10, cold_price=0.023, hot_days=7):
    # TODO
    raise NotImplementedError("TODO 8")

try:
    print([retention_plan(g, b) for g, b in ((100, 300), (100, 1200), (20, 200), (500, 100))])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 9 · Límites por tenant (TODO; con Docker se comprueba en vivo)

Escribe `tenant_limits()` → `(loki_limits, tempo_overrides)`: dicts con las claves de `limits_config` de Loki (máx. **20** streams activos, 8 etiquetas por serie, 1 MB/s de ingesta con ráfaga de 1 MB, 512 KB/s por stream) y de `overrides.defaults` de Tempo (**3 000** bytes por traza máx., 200 000 B/s de ingesta con ráfaga de 200 000).

<details>
<summary>🪜 Pista 1</summary>

Las claves están en la lección §9 y en `configs/loki-multitenant-limits.yaml` y `configs/tempo-multitenant-limits.yaml`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Loki: `{"max_streams_per_user": 20, "max_label_names_per_series": 8, "ingestion_rate_mb": 1, "ingestion_burst_size_mb": 1, "per_stream_rate_limit": "512KB"}`.

</details>

In [ ]:
def tenant_limits():
    # TODO: claves de Loki: max_streams_per_user, max_label_names_per_series, ingestion_rate_mb, ingestion_burst_size_mb, per_stream_rate_limit
    #       claves de Tempo: {"global": {"max_bytes_per_trace": ...}, "ingestion": {"rate_limit_bytes": ..., "burst_size_bytes": ...}}
    raise NotImplementedError("TODO 9")

try:
    print(tenant_limits())
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Rúbrica · ejecútala cuando quieras

In [ ]:
def run_rubric(docker=DOCKER):
    R = {}
    def chk(name, fn):
        try:
            R[name] = bool(fn())
        except NotImplementedError:
            R[name] = None
        except Exception as e:
            R[name] = False; print(f"  {name}: {type(e).__name__}: {e}")
    ref_vr = {"spans_per_trace": CORPUS.summary()["spans_per_trace"], "bytes_per_span": CORPUS.summary()["bytes_per_span"]}
    chk("R1 volume_report: spans/traza, bytes/span, fracción interesante y GB/día correctos (±1 %)", lambda: (lambda v: abs(v["spans_per_trace"] / ref_vr["spans_per_trace"] - 1) < .01 and abs(v["bytes_per_span"] / ref_vr["bytes_per_span"] - 1) < .01
        and abs(v["interesting_rate"] - CORPUS.summary()["interesting_rate"]) < 1e-9 and 0.10 < v["interesting_bytes_frac"] < 0.18 and abs(v["gb_per_day_traces"] / (1e6 * ref_vr["spans_per_trace"] * ref_vr["bytes_per_span"] / 1e9) - 1) < .01)(volume_report(CORPUS, 1e6)))
    chk("R2 baseline_for_budget: presupuesto del 25 % de los bytes → base entre 11 % y 15 %", lambda: 11.0 <= baseline_for_budget(CORPUS, 25) <= 15.0 and baseline_for_budget(CORPUS, 5) <= 0)
    def r3():
        pols = build_policies(baseline_for_budget(CORPUS, 25))
        names = [p.name for p in pols]
        res = S.TailSampler(pols, 30).run(CORPUS.traces)
        c = S.coverage(res.kept, CORPUS)
        return {"errors", "slow", "costly", "baseline"} <= set(names) and c["recall_interesting_pct"] == 100.0 and c["kept_bytes_pct"] <= 27.0
    chk("R3 build_policies: 100 % de lo interesante y ≤ 27 % de los bytes (presupuesto 25 % + ruido del hash)", r3)
    def r4():
        pols = build_policies(12.0)
        ev = evaluate(pols, CORPUS, 30.0, 16)
        return ev["dropped_too_early"] == 0 and ev["peak_traces"] <= ev["num_traces"] <= 2.2 * ev["peak_traces"] and num_traces_for(200, 30) == 9000
    chk("R4 evaluate/num_traces_for: sin descartes tempranos, num_traces entre 1× y 2,2× el pico y num_traces_for(200, 30) = 9 000", r4)
    def r5():
        b = 12.0
        kept = S.TailSampler(build_policies(b), 30).run(CORPUS.traces).kept
        truth = S.estimate_metrics(CORPUS.traces)["error_rate"]
        naive = S.estimate_metrics(kept)["error_rate"]
        est = corrected_error_rate(kept, b)
        return abs(est / truth - 1) < 0.15 and naive > 3 * truth
    chk("R5 corrected_error_rate: error relativo < 15 % frente a la verdad (el crudo está > 3× por encima)", r5)
    def r6():
        y = collector_yaml(build_policies(12.0), 9000)
        ok_text = all(k in y for k in ("tail_sampling", "span_metrics", "forward", "decision_wait", "ottl_condition")) and "numeric_attribute" not in y
        if not docker:
            return ok_text
        return ok_text and S.validate_collector_config(y)[0]
    chk("R6 collector_yaml: contiene tail_sampling + span_metrics antes del muestreo, sin numeric_attribute con decimales" + ("" if docker else " (sin Docker: solo texto)") + " y pasa `otelcol validate`", r6)
    chk("R7 cardinality_check: acepta 120 series, rechaza la tormenta y señala ticket_id", lambda: cardinality_check({"intent": 4, "tier": 3, "model": 10}, 1000) == (True, None) and cardinality_check({"intent": 4, "tier": 3, "ticket_id": 2_000_000}, 100_000) == (False, "ticket_id"))
    chk("R8 retention_plan: coincide con la referencia en 4 casos (días máximos dentro del presupuesto)", lambda: [retention_plan(g, b) for g, b in ((100, 300), (100, 1200), (20, 200), (500, 100))] == [REF_RET[(100, 300)], REF_RET[(100, 1200)], REF_RET[(20, 200)], REF_RET[(500, 100)]])
    def r9():
        if not docker:
            return None
        loki, tempo = tenant_limits()
        with S.BackendLab("loki", S.loki_lab_config(loki, auth=True)) as lk:
            now = time.time_ns()
            st = lambda n: [{"stream": {"service_name": "s", "t": f"{i}"}, "values": [[str(now + i), "x"]]} for i in range(n)]
            c_ok = lk.loki_push("a", st(10))[0]; c_bad = lk.loki_push("a", st(40))[0]
        return c_ok == 204 and c_bad == 429 and tempo["global"]["max_bytes_per_trace"] <= 5000
    chk("R9 tenant_limits (con Docker): Loki acepta 10 streams y rechaza 40 con 429; Tempo con tope de bytes por traza ≤ 5 000", r9)
    ok = sum(v is True for v in R.values()); tot = sum(v is not None for v in R.values())
    for k, v in R.items():
        print(("✅" if v else ("⏭️ " if v is None else "❌")), k)
    print(f"\nPuntuación: {ok}/{tot} comprobaciones superadas")
    return R

try:
    run_rubric()
except NameError as e:
    print("Define las funciones de los pasos primero:", e)

## Retos extra (nivel experto)

1. **Dos capas**: escribe la configuración de la capa 1 (`load_balancing` por `traceID` hacia 3 Collectors con `tail_sampling`) y valídala con `otelcol validate`.
2. **`bytes_limiting` como red de seguridad**: añade la política y simula un incidente donde el 100 % de las trazas es error: ¿cuántos bytes/s dejas pasar y cuántos errores pierdes?
3. **Muestreo por intención**: baja el baseline al 2 % para `kb_question` (la más barata de observar) y mantén el 20 % para `refund`; corrige las métricas con pesos por estrato.
4. **Logs coherentes con las trazas**: calcula qué fracción de los logs quedaría con un `trace_id` colgante si descartas la traza pero conservas los logs, y diseña la política de logs.
5. **Alerta de descartes**: escribe la regla de Grafana (módulo 16) sobre `otelcol_processor_tail_sampling_sampling_trace_dropped_too_early` > 0 durante 5 min.

## Reflexión

- ¿Qué pasaría con tu política en un **incidente** donde casi todas las trazas fueran error? ¿Quién protege tu presupuesto?
- ¿Cómo repartirías el 15 % entre equipos con tráficos 100× distintos? (tenants, cuotas, `rate_limiting`).
- ¿Qué dato necesitarías **medir** en tu sistema real antes de fiarte de las cifras de coste del notebook?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable: redefine las funciones y vuelve a lanzar la rúbrica.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def volume_report(corpus, tickets_per_day):
    sm = corpus.summary()
    inter = [t for t in corpus.traces if t.interesting]
    return {"spans_per_trace": sm["spans_per_trace"], "bytes_per_span": sm["bytes_per_span"], "interesting_rate": sm["interesting_rate"],
            "interesting_bytes_frac": sum(t.bytes for t in inter) / corpus.total_bytes,
            "gb_per_day_traces": tickets_per_day * sm["spans_per_trace"] * sm["bytes_per_span"] / 1e9}

def baseline_for_budget(corpus, budget_bytes_pct):
    vr = volume_report(corpus, 1)
    i = vr["interesting_bytes_frac"]
    b = (budget_bytes_pct / 100 - i) / (1 - i)
    return max(0.0, math.floor(b * 1000) / 10)          # en % con un decimal, hacia abajo

def build_policies(baseline_pct):
    return [S.StatusCode(("ERROR",), name="errors"), S.Latency(5000, name="slow"),
            S.AttrCompare("supportops.cost_usd", ">", 0.0005, name="costly"), S.Probabilistic(baseline_pct, name="baseline")]

def num_traces_for(traces_per_s, decision_wait_s, margin=1.5):
    return int(math.ceil(traces_per_s * decision_wait_s * margin))

def evaluate(policies, corpus, decision_wait_s=30.0, rate_factor=16):
    rescaled = S.rescale_rate(corpus.traces, rate_factor)
    peak = S.TailSampler(policies, decision_wait_s, num_traces=10**9).run(rescaled).peak_traces
    nt = num_traces_for(rate_factor * len(corpus) / corpus.duration_s, decision_wait_s)
    r = S.TailSampler(policies, decision_wait_s, num_traces=nt).run(rescaled)
    return {**S.coverage(r.kept, corpus), "peak_traces": peak, "num_traces": nt, "dropped_too_early": len(r.dropped_too_early), "kept": r.kept}

def corrected_error_rate(kept, baseline_pct):
    keep_all = lambda t: t.interesting
    w = S.inclusion_weights(kept, keep_all, baseline_pct / 100)
    return S.estimate_metrics(kept, w)["error_rate"]

def collector_yaml(policies, num_traces):
    return S.collector_config(policies, decision_wait="30s", num_traces=num_traces)

def cardinality_check(dims, max_series):
    total = S.series_estimate(dims)
    if total <= max_series:
        return True, None
    return False, max(dims, key=dims.get)

def retention_plan(traces_gb_day, budget_usd_month, hot_price=0.10, cold_price=0.023, hot_days=7):
    best = 0
    for d in range(1, 366):
        cost = traces_gb_day * (min(d, hot_days) * hot_price + max(0, d - hot_days) * cold_price)
        if cost <= budget_usd_month:
            best = d
    return best

def tenant_limits():
    loki = {"max_streams_per_user": 20, "max_label_names_per_series": 8, "ingestion_rate_mb": 1, "ingestion_burst_size_mb": 1, "per_stream_rate_limit": "512KB"}
    tempo = {"global": {"max_bytes_per_trace": 3000}, "ingestion": {"rate_limit_bytes": 200000, "burst_size_bytes": 200000}}
    return loki, tempo

VR = volume_report(CORPUS, TICKETS_DAY)
b = baseline_for_budget(CORPUS, BUDGET_BYTES_PCT)
print({k: round(v, 4) for k, v in VR.items()}, "· baseline:", b)
ev = evaluate(build_policies(b), CORPUS)
print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in ev.items() if k != "kept"})
kept = ev["kept"]
print("tasa de error: verdad", round(S.estimate_metrics(CORPUS.traces)["error_rate"], 4), "· cruda", round(S.estimate_metrics(kept)["error_rate"], 4), "· corregida", round(corrected_error_rate(kept, b), 4))
cm = S.StackCostModel.from_corpus(CORPUS, TICKETS_DAY, spans_per_trace=40, bytes_per_span=2500, trace_keep=ev["kept_bytes_pct"] / 100)
print("días de retención de trazas dentro de", BUDGET_USD_MONTH, "USD/mes (escenario supuesto de 40 spans):", retention_plan(cm.daily_gb()["traces_stored"], BUDGET_USD_MONTH))
R = run_rubric()